# Credit Default Prediction - Machine Learning Experiments

This notebook is the project report for the credit default prediction ML work. It uses the real German Credit dataset stored in `ml/dataset/german_credit_data.csv` and generates the artifacts consumed by the .NET backend and React frontend.

## 1. Introduction

The business goal is to estimate credit default risk before a loan decision is made. The target variable is `kredit`: value `0` represents bad/default-risk credit and value `1` represents good/non-default credit. Because bad credit is the minority class, the experiments use a stratified train/test split and evaluate recall, precision, F1-score, ROC-AUC, and confusion matrices, not accuracy alone.

## 2. Dataset Description

The dataset contains 1,000 German credit applications. Important fields include checking account status (`laufkont`), credit duration (`laufzeit`), credit history (`moral`), purpose (`verw`), credit amount (`hoehe`), savings status (`sparkont`), employment duration (`beszeit`), installment rate (`rate`), property category (`verm`), age (`alter`), housing (`wohn`), existing credits (`bishkred`), job type (`beruf`), and foreign worker indicator (`gastarb`). These fields describe applicant financial stability, loan characteristics, and repayment history.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('..') / 'dataset' / 'german_credit_data.csv'
df = pd.read_csv(DATA_PATH)
df.shape, df.head()

((1000, 21),
    laufkont  laufzeit  moral  verw  hoehe  sparkont  beszeit  rate  famges  \
 0         1        18      4     2   1049         1        2     4       2   
 1         1         9      4     0   2799         1        3     2       3   
 2         2        12      2     9    841         2        4     2       2   
 3         1        12      4     0   2122         1        3     3       3   
 4         1        12      4     0   2171         1        3     4       3   
 
    buerge  ...  verm  alter  weitkred  wohn  bishkred  beruf  pers  telef  \
 0       1  ...     2     21         3     1         1      3     2      1   
 1       1  ...     1     36         3     1         2      3     1      1   
 2       1  ...     1     23         3     1         1      2     2      1   
 3       1  ...     1     39         3     1         2      2     1      1   
 4       1  ...     2     38         1     2         2      2     2      1   
 
    gastarb  kredit  
 0        2       1

In [2]:
missing_values = df.isna().sum()
class_distribution = df['kredit'].value_counts().sort_index()
missing_values, class_distribution

(laufkont    0
 laufzeit    0
 moral       0
 verw        0
 hoehe       0
 sparkont    0
 beszeit     0
 rate        0
 famges      0
 buerge      0
 wohnzeit    0
 verm        0
 alter       0
 weitkred    0
 wohn        0
 bishkred    0
 beruf       0
 pers        0
 telef       0
 gastarb     0
 kredit      0
 dtype: int64,
 kredit
 0    300
 1    700
 Name: count, dtype: int64)

## 3. Data Preprocessing

The source file uses numeric German Credit encodings for both ordinal and categorical fields. Categorical variables still need consistent numeric treatment for scikit-learn estimators. Scaling is required for distance-based and gradient-based models such as KNN, Logistic Regression, SVM, and MLP because those models are sensitive to feature magnitude. Tree-based models such as Decision Tree and Random Forest do not require scaling, but the same train/test split is used for fair comparison.

In [3]:
target = 'kredit'
X = df.drop(columns=[target])
y = df[target]
categorical_features = ['laufkont', 'moral', 'verw', 'sparkont', 'beszeit', 'famges', 'buerge', 'verm', 'weitkred', 'wohn', 'beruf', 'pers', 'telef', 'gastarb']
numerical_features = [col for col in X.columns if col not in categorical_features]
categorical_features, numerical_features

(['laufkont',
  'moral',
  'verw',
  'sparkont',
  'beszeit',
  'famges',
  'buerge',
  'verm',
  'weitkred',
  'wohn',
  'beruf',
  'pers',
  'telef',
  'gastarb'],
 ['laufzeit', 'hoehe', 'rate', 'wohnzeit', 'alter', 'bishkred'])

## 4. Train/Test Split

The target is imbalanced: there are fewer bad/default-risk cases than good/non-default cases. A stratified split preserves this ratio in both training and test sets so each model is evaluated against the same realistic distribution.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.value_counts(normalize=True).sort_index(), y_test.value_counts(normalize=True).sort_index()

((800, 20),
 (200, 20),
 kredit
 0    0.3
 1    0.7
 Name: proportion, dtype: float64,
 kredit
 0    0.3
 1    0.7
 Name: proportion, dtype: float64)

## 5. Classification Models

The project compares classifiers with different assumptions: Logistic Regression as a linear model, K-Nearest Neighbors as a distance-based model, Decision Tree as an interpretable tree model, Random Forest as an ensemble, MLPClassifier as a neural network, and Support Vector Machine as an optional margin-based classifier. These models are suitable for credit risk because they capture different types of relationships between applicant attributes and default risk.

## 6. Hyperparameter Tuning

Each classifier is tuned with GridSearchCV on the same training set. The scoring objective is F1-score for the bad-credit class (`kredit = 0`) because missing a risky applicant is more expensive than optimizing overall accuracy. The tested grids include C/solver for Logistic Regression, neighbors/weights/metric for KNN, depth/split/leaf/criterion for Decision Tree, estimators/depth/split/leaf/features for Random Forest, hidden layers/activation/alpha/learning rate for MLP, and C/kernel/gamma for SVM.

In [5]:
%run ../train_model.py

C:\Users\oltsh\Desktop\credit-default-prediction\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


C:\Users\oltsh\Desktop\credit-default-prediction\ml\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\oltsh\Desktop\credit-default-prediction\ml\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


{
  "dataset_shape": [
    1000,
    21
  ],
  "missing_values": {
    "laufkont": 0,
    "laufzeit": 0,
    "moral": 0,
    "verw": 0,
    "hoehe": 0,
    "sparkont": 0,
    "beszeit": 0,
    "rate": 0,
    "famges": 0,
    "buerge": 0,
    "wohnzeit": 0,
    "verm": 0,
    "alter": 0,
    "weitkred": 0,
    "wohn": 0,
    "bishkred": 0,
    "beruf": 0,
    "pers": 0,
    "telef": 0,
    "gastarb": 0,
    "kredit": 0
  },
  "numerical_features": [
    "laufkont",
    "laufzeit",
    "moral",
    "verw",
    "hoehe",
    "sparkont",
    "beszeit",
    "rate",
    "famges",
    "buerge",
    "wohnzeit",
    "verm",
    "alter",
    "weitkred",
    "wohn",
    "bishkred",
    "beruf",
    "pers",
    "telef",
    "gastarb"
  ],
  "categorical_features": [
    "laufkont",
    "moral",
    "verw",
    "sparkont",
    "beszeit",
    "famges",
    "buerge",
    "verm",
    "weitkred",
    "wohn",
    "beruf",
    "pers",
    "telef",
    "gastarb"
  ],
  "target": "kredit",
  "class_distribu

## 7. Model Evaluation And Comparison

The training script saves `model_comparison_results.csv` and `model_comparison_results.json`. The table includes tested hyperparameters, best hyperparameters, cross-validation score, test accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix values for every tuned classifier.

In [6]:
results = pd.read_json(Path('..') / 'results' / 'model_comparison_results.json')
results[['model', 'cv_score', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'roc_auc', 'best_hyperparameters']]

,model,cv_score,test_accuracy,test_precision,test_recall,test_f1,roc_auc,best_hyperparameters
0,Logistic Regression,0.593960,0.760,0.575000,0.766667,0.657143,0.825952,"{""C"": 0.01, ""penalty"": ""l2"", ""solver"": ""lbfgs""}"
1,Random Forest,0.616177,0.765,0.591549,0.700000,0.641221,0.818690,"{""max_depth"": 5, ""max_features"": ""sqrt"", ""min_..."
2,Support Vector Machine,0.599089,0.735,0.544304,0.716667,0.618705,0.822857,"{""C"": 1.0, ""gamma"": ""scale"", ""kernel"": ""rbf""}"
3,Neural Network,0.550857,0.780,0.681818,0.500000,0.576923,0.815595,"{""activation"": ""tanh"", ""alpha"": 0.0001, ""hidde..."
4,Decision Tree,0.573053,0.675,0.471910,0.700000,0.563758,0.698393,"{""criterion"": ""gini"", ""max_depth"": 5, ""min_sam..."
5,K-Nearest Neighbors,0.479631,0.710,0.527778,0.316667,0.395833,0.732440,"{""metric"": ""manhattan"", ""n_neighbors"": 5, ""wei..."


## 8. Feature Selection

Two feature reduction methods are evaluated: SelectKBest with ANOVA F-score and Random Forest feature importance. The experiments compare all features against top 5, top 10, and top 15 feature subsets. This shows whether simpler input improves generalization or removes useful signal.

In [7]:
feature_selection = pd.read_json(Path('..') / 'results' / 'feature_selection_results.json')
feature_importance = pd.read_csv(Path('..') / 'results' / 'feature_importance.csv')
feature_selection[['method', 'k', 'test_accuracy', 'test_recall', 'test_f1']].head(), feature_importance.head(10)

(                     method   k  test_accuracy  test_recall   test_f1
 0               SelectKBest  10          0.770     0.766667  0.666667
 1               SelectKBest   5          0.740     0.800000  0.648649
 2               SelectKBest  15          0.740     0.750000  0.633803
 3  Random Forest Importance  15          0.805     0.483333  0.597938
 4  Random Forest Importance  10          0.790     0.466667  0.571429,
     feature                            description  importance
 0     hoehe                          credit amount    0.125817
 1  laufkont                checking account status    0.124631
 2  laufzeit              credit duration in months    0.102063
 3     alter                           age in years    0.100510
 4      verw                           loan purpose    0.061352
 5  sparkont                 savings account status    0.057465
 6     moral                         credit history    0.052122
 7   beszeit                    employment duration    0.0497

## 9. Neural Network Architectures

The input layer represents the 20 applicant and loan features. Hidden layers transform those inputs into learned representations. ReLU and tanh activations introduce non-linear behavior, while the output predicts the final credit class. Larger architectures can learn more complex patterns but may overfit, especially on a 1,000-row dataset.

In [8]:
nn_results = pd.read_json(Path('..') / 'results' / 'neural_network_results.json')
nn_results[['architecture', 'hidden_layer_sizes', 'activation', 'training_time_seconds', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

,architecture,hidden_layer_sizes,activation,training_time_seconds,accuracy,precision,recall,f1,roc_auc
0,Architecture 1,"[64, 32]",relu,0.174758,0.770,0.659091,0.483333,0.557692,0.826786
1,Architecture 2,"[128, 64, 32]",relu,0.261085,0.795,0.731707,0.500000,0.594059,0.807738
2,Architecture 3,"[32, 16]",tanh,0.211173,0.780,0.681818,0.500000,0.576923,0.815595


## 10. Clustering

KMeans is trained without the target label because clustering is unsupervised. The label is used only after fitting to compare clusters with known credit classes. The experiments test k = 2, 3, 4, and 5, and evaluate inertia, silhouette score, and Adjusted Rand Index. Clusters may not perfectly match default/non-default labels because repayment risk is a supervised outcome, while KMeans groups geometric similarity in feature space.

In [9]:
clustering = pd.read_json(Path('..') / 'results' / 'clustering_results.json')
clustering

,k,silhouette_score,inertia,adjusted_rand_index
0,2,0.083122,18348.891282,-0.000628
1,3,0.074995,17326.174859,0.048085
2,4,0.083834,16330.154094,0.035625
3,5,0.089571,15638.251019,0.035688


## 11. Best Model Discussion

The selected model is the tuned classifier with the highest bad-credit F1-score on the shared test set. This criterion balances precision and recall for risky applicants. The final model, preprocessing pipeline, feature columns, and metadata are saved in `ml/models` for backend prediction.

In [10]:
import json
metadata = json.loads((Path('..') / 'models' / 'model_metadata.json').read_text())
metadata

{'selected_model_name': 'Logistic Regression',
 'training_date': '2026-06-28T18:20:05.090011+00:00',
 'dataset_name': 'German Credit dataset',
 'target': 'kredit',
 'target_mapping': {'0': 'bad/default risk', '1': 'good/non-default'},
 'feature_count': 20,
 'feature_columns': ['laufkont',
  'laufzeit',
  'moral',
  'verw',
  'hoehe',
  'sparkont',
  'beszeit',
  'rate',
  'famges',
  'buerge',
  'wohnzeit',
  'verm',
  'alter',
  'weitkred',
  'wohn',
  'bishkred',
  'beruf',
  'pers',
  'telef',
  'gastarb'],
 'metrics': {'accuracy': 0.76,
  'precision': 0.575,
  'recall': 0.7666666666666667,
  'f1': 0.6571428571428571,
  'roc_auc': 0.825952380952381},
 'best_hyperparameters': {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'},
 'selected_features': ['hoehe',
  'laufkont',
  'laufzeit',
  'alter',
  'verw',
  'sparkont',
  'moral',
  'beszeit',
  'verm',
  'rate']}

## 12. Final Conclusion

The project now includes reproducible preprocessing, stratified train/test evaluation, six classifier families, hyperparameter tuning for every classifier, feature selection experiments, neural network architecture comparison, and KMeans clustering. Limitations include the small dataset size, encoded categorical variables without original text labels, and the need for future calibration/fairness checks. Future improvements could add probability calibration, threshold tuning, richer loan-application features, and monitoring for drift after deployment.

## References

- German Credit dataset distributed with this project.
- scikit-learn documentation for classifiers, GridSearchCV, SelectKBest, RandomForest feature importance, MLPClassifier, KMeans, PCA, and model evaluation metrics.